# 01 — Honest baseline

The original notebook (`Choi_Final.ipynb`) reports 97.69% Random Forest accuracy under a **random** 80/20 split. 47% of rows are duplicate feature vectors, so many test rows are copies of training rows. This notebook re-evaluates every model with `StratifiedGroupKFold` on pattern-group ids so a feature vector never appears on both sides of a fold.

It also adds XGBoost and LightGBM, tunes under grouped CV, picks a max-F1 threshold, and checks probability calibration.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src"))

import pandas as pd
from phishing.config import DEPLOYABLE_FEATURES, REPORTS_DIR
from phishing.data import grouped_split, load_xy, unique_pattern_stats, load_raw
from phishing.evaluate import (
    best_f1_threshold, calibrate, fpr_target_threshold,
    leakage_delta_table, metric_dict, threshold_report,
)
from phishing.models import ALL_MODELS, build_model

pd.set_option("display.float_format", "{:.4f}".format)
print(unique_pattern_stats(load_raw()))

## Leakage-delta table

Random 5-fold CV vs grouped 5-fold CV. The accuracy gap is the optimism induced by duplicate patterns. Re-run `PYTHONPATH=src python -m phishing.cli evaluate` to regenerate `reports/leakage_delta.csv` if it is missing.

In [ ]:
delta_path = REPORTS_DIR / "leakage_delta.csv"
if delta_path.exists():
    delta = pd.read_csv(delta_path, index_col=0)
else:
    X, y, groups = load_xy()
    delta = leakage_delta_table(X, y, groups, model_names=ALL_MODELS)
    REPORTS_DIR.mkdir(parents=True, exist_ok=True)
    delta.to_csv(delta_path)
delta

## Grouped holdout, thresholds, calibration

The test split is grouped as well — a random 80/20 would re-introduce the same leak. After fitting, we search a max-F1 threshold and a 1% FPR operating point, then compare Brier scores before and after isotonic calibration.

In [ ]:
X, y, groups = load_xy()
X_tr, X_te, y_tr, y_te, g_tr, g_te = grouped_split(X, y, groups)
print(f"train={len(X_tr)} test={len(X_te)} disjoint groups={set(g_tr).isdisjoint(g_te)}")

rows = []
for name in ALL_MODELS:
    est = build_model(name)
    est.fit(X_tr, y_tr)
    proba = est.predict_proba(X_te)[:, 1]
    pred = (proba >= 0.5).astype(int)
    m = metric_dict(y_te, pred, proba)
    t_f1, _ = best_f1_threshold(y_te, proba)
    t_fpr, fpr = fpr_target_threshold(y_te, proba, max_fpr=0.01)
    m.update({"model": name, "f1_threshold": t_f1, "fpr01_threshold": t_fpr, "fpr_at_target": fpr})
    rows.append(m)
pd.DataFrame(rows).set_index("model")

The original write-up noted Gradient Boosting's AUROC nearly matching Random Forest while accuracy trailed by ~2 points. That is a **threshold** problem: the default 0.5 cut is not the F1-optimal point. The table above reports both the 0.5-cut metrics and the searched thresholds.

Train the deployable 25-feature model with:

```bash
PYTHONPATH=src python -m phishing.cli train --tune
```